In [8]:
#imports
import pandas as pd
import numpy as np
from scipy import stats
from IPython.display import display

In [9]:
pd.set_option('display.max_columns', None)

# get the dataset as a dataframe 
df = pd.read_csv('MDS sample.csv')
# change dates into datetime datatypes
df['DATE_ASSESSMENT'] = pd.to_datetime(df['DATE_ASSESSMENT'], format='%d%b%Y', errors='coerce')
df['BENE_DEATH_DT'] = pd.to_datetime(df['BENE_DEATH_DT'], format='%d%b%Y',errors='coerce')
# change weight recordings of 0 into NaN
df['WT_OLD'] = df['WT_OLD'].replace(0, np.nan)
# sort the assessments of each person by the date
df_sorted = df.sort_values(by=['BENE_ID', 'DATE_ASSESSMENT'], ascending=[True, False]).reset_index(drop=True)

# convert the values in RACE column so that it shows the race instead of the corresponding number
race_map = {
    6: "Native Hawaiian/Pacific Islander",
    5: "American Indian/Alaskan Native",
    4: "Asian",
    3: "Black",
    2: "Hispanic",
    1: "White",
    0: "Missing"
}
df_sorted["RACE"] = df_sorted["RACE"].map(race_map)

#c onvert values in GENDER column to actual genders instead of A and B
gender_map = {
    'A': 'Male',
    'B': 'Female'
}
df_sorted["GENDER"] = df_sorted["GENDER"].map(gender_map)

#find row with earliest assessment for each patient
first_assessment = (
    df_sorted
    .sort_values(["BENE_ID", "DATE_ASSESSMENT"], ascending=[True, True])
    .drop_duplicates(subset="BENE_ID", keep="first")
    .reset_index(drop=True)
)

# Cohort selection / cleaning table

cohort_steps = []

def add_cohort_step(step_name, before_ids, after_ids):
    """
    Records patient-level counts for one cleaning step.
    before_ids and after_ids should be BENE_ID Series/list/index objects.
    """
    before_set = set(before_ids)
    after_set = set(after_ids)

    cohort_steps.append({
        "Cohort selection": step_name,
        "Number of selected patients": len(after_set),
        "Dropped from the above step": len(before_set - after_set)
    })

# Start with all unique residents based on first assessment
current = first_assessment.copy()

cohort_steps.append({
    "Cohort selection": "All residents with an identifiable first assessment",
    "Number of selected patients": current["BENE_ID"].nunique(),
    "Dropped from the above step": 0
})

# Step 1: age filter
before_ids = current["BENE_ID"]
current = current[current["AGE"] >= 66].copy()
add_cohort_step(
    "Age 66 years or older at first assessment",
    before_ids,
    current["BENE_ID"]
)

# Step 2: valid admission weight
before_ids = current["BENE_ID"]
current = current[current["WT_OLD"].notna()].copy()
add_cohort_step(
    "With valid weight at first assessment",
    before_ids,
    current["BENE_ID"]
)

# Step 3: remove extreme admission weights
low, high = first_assessment["WT_OLD"].quantile([0.01, 0.99])

before_ids = current["BENE_ID"]
current = current[current["WT_OLD"].between(low, high)].copy()
add_cohort_step(
    "No extreme weight at first assessment, 1st–99th percentile",
    before_ids,
    current["BENE_ID"]
)

# Step 4: valid BMI
before_ids = current["BENE_ID"]
current = current[current["BMI"].notna()].copy()
add_cohort_step(
    "With valid BMI at first assessment",
    before_ids,
    current["BENE_ID"]
)

# Step 5: obese BMI group
before_ids = current["BENE_ID"]
current = current[current["BMI"] >= 30].copy()
add_cohort_step(
    "BMI over 30 at first assessment",
    before_ids,
    current["BENE_ID"]
)

# Final valid patients
valid_patients = current["BENE_ID"]

# Keep all assessment rows for the final selected patient cohort
df_sorted = df_sorted[df_sorted["BENE_ID"].isin(valid_patients)].copy()

# Create table
cohort_selection_table = pd.DataFrame(cohort_steps)

display(cohort_selection_table)

# Optional: save table
cohort_selection_table.to_csv(
    "Visualizations_and_Tables/cohort_selection_table.csv",
    index=False
)



# load in data for a specific subject
# df_sorted[df_sorted['BENE_ID'].str.contains('JJJJJJ4J4WVppzz')]

#df_sorted[df_sorted['BENE_ID'].str.contains('JJJJJJJM3zMz3oo')].sort_values(["BENE_ID", "DATE_ASSESSMENT"]).groupby("BENE_ID").first().reset_index()
#df_sorted[df_sorted['BENE_ID'].str.contains('JJJJJJJozWpzopV')]
#len(df_sorted)

/var/folders/0y/mmp_lnms4tx1tvdz0cwsqdx40000gs/T/ipykernel_39901/964456703.py:4: DtypeWarning: Columns (0: GG0130E1_BTHE_SELF_STRT_CD, 1: GG0130E3_BTHE_SELF_END_CD, 2: GG0130F1_UPR_DRSNG_STRT_CD, 3: GG0130F3_UPR_DRSNG_END_CD, 4: GG0130G1_LWR_DRSNG_STRT_CD, 5: GG0130G3_LWR_DRSNG_END_CD) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('MDS sample.csv')


,Cohort selection,Number of selected patients,Dropped from the above step
0,All residents with an identifiable first asses...,5000,0
1,Age 66 years or older at first assessment,4119,881
2,With valid weight at first assessment,1330,2789
3,"No extreme weight at first assessment, 1st–99t...",1320,10
4,With valid BMI at first assessment,1314,6
5,BMI over 30 at first assessment,430,884


In [10]:
# define summary function for each person
def summarize_patient(group):

    # list of illness columns
    illness_cols = [
    'ARTHRITIS','STROKE','CANCER','COPD','DIAB','DEMENTIA','DEPRESSION',
    'HEARTFAILURE','HYPERTENSION','ESRD','FT_PROBLEM','HEARING',
    'BOWEL_INCONTINENCE','URINE_INCONTINENCE','ANXTY','MNC_DPRSN','SCHZOPRNIA','PRESSURE_ULCER'
    ]

    first_date = group['DATE_ASSESSMENT'].min()
    last_date = group['DATE_ASSESSMENT'].max()
    death_date = group['BENE_DEATH_DT'].min()
    months_between_assessments = (last_date.year - first_date.year) * 12 + (last_date.month - first_date.month)
    assessment1_death_between = (death_date.year - first_date.year) * 12 + (death_date.month - first_date.month) if death_date != np.nan else np.nan

    init_wt = group['WT_OLD'].dropna().iloc[-1] if group['WT_OLD'].notna().any() else np.nan
    end_wt =  group['WT_OLD'].dropna().iloc[0] if group['WT_OLD'].notna().any() else np.nan

    summary = {
        'INITIAL_AGE': group.loc[group['DATE_ASSESSMENT'] == first_date, 'AGE'].values[0],
        'BENE_DEATH_DT': group['BENE_DEATH_DT'].dropna().iloc[-1] if group['BENE_DEATH_DT'].notna().any() else np.nan,
        'GENDER': group['GENDER'].dropna().iloc[-1] if group['GENDER'].notna().any() else np.nan,
        'RACE': group['RACE'].dropna().iloc[-1] if group['RACE'].notna().any() else np.nan,
        'First_Assessment': first_date,
        'Last_Assessment': last_date,
        'Months_Between_Assessments': months_between_assessments,
        'assessment1_death_month_between': assessment1_death_between,
        'ADL_SCORE_INIT': group['ADL_SCORE'].dropna().iloc[-1] if group['ADL_SCORE'].notna().any() else np.nan,
        'MOBILITY_SP1_INIT':  group['MOBILITY_SP1'].dropna().iloc[-1] if group['MOBILITY_SP1'].notna().any() else np.nan,
        'COG_MS1_INIT':  group['COG_MS1'].dropna().iloc[-1] if group['COG_MS1'].notna().any() else np.nan,
        'Initial_BIMS': group['BIMS'].dropna().iloc[-1] if group['BIMS'].notna().any() else np.nan,
        'Initial_BMI': group['BMI'].dropna().iloc[-1] if group['BMI'].notna().any() else np.nan,
        'Initial_WT': init_wt,
        'End_WT': end_wt,
        'Overall_WT_Change': end_wt - init_wt,
        'Overall_WT_Change_Percent' : (end_wt - init_wt) / init_wt if group['WT_OLD'].notna().any() else np.nan
    }

    # illness present if any row has a non-null value (i checked and 
    # each col only contains either NaN or the name of the illness)
    for col in illness_cols:
        illness_present = group[col].notna().any()
        summary[col] = int(illness_present)

    # maximum weight loss experienced going from 1 assessment to the next (%)
    wt_series = group.sort_values('DATE_ASSESSMENT')['WT_OLD'].dropna()
    if len(wt_series) >= 2:
        # initial weight for denominator
        initial_wt = wt_series.iloc[0]
        # weight changes between successive assessments (negative = loss)
        wt_diffs = wt_series.diff()
        # largest (most negative) drop in kg
        max_drop_kg = wt_diffs.min()
        if pd.notna(initial_wt) and initial_wt > 0:
            summary['MOST_WT_LOSS'] = (max_drop_kg / initial_wt) * 100   # now stored as %
        else:
            summary['MOST_WT_LOSS'] = np.nan
    else:
        summary['MOST_WT_LOSS'] = np.nan

        # GG0130 END codes — map each to a person's most recent non-missing, 
    # performance-only value (codes 01–06); exclude non-performance codes
    gg0130_end_cols = {
        'EATG_ABILITY':  'GG0130A3_EATG_ABILITY_END_CD',
        'ORAL_ABILITY':  'GG0130B3_ORAL_ABILITY_END_CD',
        'TOILT_ABILITY': 'GG0130C3_TOILT_ABILITY_END_CD',
        'BTHE_SELF':     'GG0130E3_BTHE_SELF_END_CD',
        'UPR_DRSNG':     'GG0130F3_UPR_DRSNG_END_CD',
        'LWR_DRSNG':     'GG0130G3_LWR_DRSNG_END_CD',
    }

    VALID_PERF_CODES = {'01', '02', '03', '04', '05', '06',
                        '1', '2', '3', '4', '5', '6',
                        1.0, 2.0, 3.0, 4.0, 5.0, 6.0}

    for summary_key, col in gg0130_end_cols.items():
        if col not in group.columns:
            summary[summary_key] = np.nan
            continue

        # Sort so that the most recent assessment is first
        sorted_group = group.sort_values('DATE_ASSESSMENT', ascending=False)
        col_series = sorted_group[col]

        # Find the first (most recent) valid performance code
        val = np.nan
        for raw in col_series:
            if pd.isna(raw) or raw == '-':
                continue
            # Normalize: convert float-like values to string integer (e.g. 6.0 -> '06')
            try:
                normalized = str(int(float(raw))).zfill(2)
            except (ValueError, TypeError):
                normalized = str(raw).strip()
            if normalized in {'01','02','03','04','05','06'}:
                val = int(normalized)
                break

        summary[summary_key] = val
        
    return pd.Series(summary)

# apply group-by logic
summary_df = df_sorted.groupby('BENE_ID').apply(summarize_patient).reset_index()

summary_df

,BENE_ID,INITIAL_AGE,BENE_DEATH_DT,GENDER,RACE,First_Assessment,Last_Assessment,Months_Between_Assessments,assessment1_death_month_between,ADL_SCORE_INIT,MOBILITY_SP1_INIT,COG_MS1_INIT,Initial_BIMS,Initial_BMI,Initial_WT,End_WT,Overall_WT_Change,Overall_WT_Change_Percent,ARTHRITIS,STROKE,CANCER,COPD,DIAB,DEMENTIA,DEPRESSION,HEARTFAILURE,HYPERTENSION,ESRD,FT_PROBLEM,HEARING,BOWEL_INCONTINENCE,URINE_INCONTINENCE,ANXTY,MNC_DPRSN,SCHZOPRNIA,PRESSURE_ULCER,MOST_WT_LOSS,EATG_ABILITY,ORAL_ABILITY,TOILT_ABILITY,BTHE_SELF,UPR_DRSNG,LWR_DRSNG
0,JJJJJJ43zVVWMVo,70,NaT,Female,Black,2018-01-26,2020-01-17,24,NaN,16.0,1.0,NaN,Cognitively intact,37.105089,101.151016,97.975872,-3.175144,-0.031390,0,0,0,1,1,1,1,1,1,0,1,0,0,0,1,0,0,0,-5.829596,NaN,NaN,NaN,NaN,NaN,NaN
1,JJJJJJ4J4WVppzz,72,NaT,Male,White,2018-01-19,2020-02-05,25,NaN,6.0,1.0,1.0,Mildly impaired,35.143027,114.305184,104.779752,-9.525432,-0.083333,0,0,0,1,1,0,1,1,1,1,1,0,1,1,0,1,0,0,-12.301587,6.0,6.0,6.0,4.0,6.0,6.0
2,JJJJJJ4JJ44WMS4,74,2020-10-10,Female,White,2019-11-14,2020-03-16,4,11.0,22.0,3.0,2.0,Moderately impaired,31.580078,83.460928,88.450440,4.989512,0.059783,1,1,0,0,1,1,1,1,1,0,1,0,1,1,0,1,0,0,-1.086957,5.0,4.0,2.0,2.0,4.0,3.0
3,JJJJJJ4JJ4oV3SM,73,NaT,Female,White,2018-01-04,2018-05-25,4,NaN,15.0,3.0,2.0,Moderately impaired,31.455775,78.017824,79.832192,1.814368,0.023256,0,0,0,1,0,0,1,1,1,0,1,0,1,1,1,0,0,0,-2.325581,6.0,6.0,4.0,NaN,NaN,NaN
4,JJJJJJ4JJoopJzM,69,NaT,Female,White,2018-02-13,2020-03-30,25,NaN,18.0,2.0,2.0,Moderately impaired,39.446111,91.625584,90.718400,-0.907184,-0.009901,1,0,0,0,1,1,1,0,1,0,1,0,1,1,0,0,0,0,-9.900990,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
425,JJJJJJzp33J3SMz,71,2018-09-18,Male,Black,2018-01-17,2018-07-04,6,8.0,13.0,1.0,3.0,Severely impaired,33.245638,93.439952,90.718400,-2.721552,-0.029126,0,0,0,0,1,1,1,0,1,0,1,0,1,1,0,0,1,0,-1.941748,NaN,NaN,NaN,NaN,NaN,NaN
426,JJJJJJzpMM4Mzpz,66,NaT,Male,White,2018-02-07,2020-03-03,25,NaN,22.0,3.0,0.0,Mildly impaired,32.535035,97.068688,98.429464,1.360776,0.014019,1,0,0,0,0,0,0,1,1,0,1,0,1,1,0,0,0,0,-2.336449,NaN,NaN,NaN,NaN,NaN,NaN
427,JJJJJJzpSSzzMz4,71,2021-01-20,Female,White,2018-01-03,2020-03-16,26,36.0,7.0,1.0,2.0,Cognitively intact,31.147612,87.543256,94.347136,6.803880,0.077720,1,0,0,0,1,1,0,1,1,1,1,0,1,1,0,0,0,0,-9.326425,6.0,6.0,6.0,5.0,6.0,5.0
428,JJJJJJzpSzM3ppJ,69,2020-06-29,Female,White,2018-01-08,2020-03-08,26,29.0,18.0,3.0,2.0,Moderately impaired,35.479188,87.996848,71.667536,-16.329312,-0.185567,1,1,0,0,1,1,1,1,1,1,1,0,1,1,0,0,0,0,-14.948454,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# Table 1-style summary table:
# All Obesity + Obesity Classes, split by Alive vs Death


def create_table1_mortality_by_obesity(summary_df):
    df = summary_df.copy()

    # Basic derived variables
    df["died"] = df["BENE_DEATH_DT"].notna().astype(int)

    # Obesity classes
    # right=False means:
    # Class I: [30, 35), Class II: [35, 40), Class III: >= 40
    df["Obesity_Class"] = pd.cut(
        df["Initial_BMI"],
        bins=[30, 35, 40, np.inf],
        labels=["Class I Obesity", "Class II Obesity", "Class III Obesity"],
        right=False
    )

    # Age groups to mimic the example table
    df["Age_Group_Table1"] = pd.cut(
        df["INITIAL_AGE"],
        bins=[64, 74, 84, np.inf],
        labels=["65-74", "75-84", ">=85"],
        right=True
    )

    # ADL categories
    df["ADL_Category_Table1"] = pd.cut(
        pd.to_numeric(df["ADL_SCORE_INIT"], errors="coerce"),
        bins=[-np.inf, 12, 20, 28],
        labels=["0-12", "13-20", "21-28"],
        right=True
    )

    # Mobility categories, only if MOBILITY_SP1_INIT exists
    if "MOBILITY_SP1_INIT" in df.columns:
        mobility_map = {
            0: "None",
            1: "Mild",
            2: "Moderate",
            3: "Severe"
        }
        df["Mobility_Category_Table1"] = (
            pd.to_numeric(df["MOBILITY_SP1_INIT"], errors="coerce")
            .round()
            .map(mobility_map)
        )

    # Cognitive categories
    # This notebook already has text BIMS categories.
    # This keeps those labels if they already exist.
    if "Initial_BIMS" in df.columns:
        df["BIMS_Category_Table1"] = df["Initial_BIMS"]

    # Race cleanup: works whether RACE is coded as numbers or already labels
    race_map = {
        1: "White",
        2: "Hispanic",
        3: "Black",
        4: "Asian",
        5: "American Indian/Alaskan Native",
        6: "Native Hawaiian/Pacific Islander",
        0: "Missing"
    }

    if "RACE" in df.columns:
        df["Race_Table1"] = df["RACE"].replace(race_map)

        df["Race_Table1"] = df["Race_Table1"].replace({
            "Asian": "Asian, American Indian/Alaskan Native, Native Hawaiian/Pacific Islander",
            "American Indian/Alaskan Native": "Asian, American Indian/Alaskan Native, Native Hawaiian/Pacific Islander",
            "Native Hawaiian/Pacific Islander": "Asian, American Indian/Alaskan Native, Native Hawaiian/Pacific Islander"
        })

    # Gender cleanup
    if "GENDER" in df.columns:
        df["Gender_Table1"] = df["GENDER"].replace({
            1: "Male",
            2: "Female",
            "M": "Male",
            "F": "Female",
            "m": "Male",
            "f": "Female"
        })

    # Helper functions
    def fmt_n_pct(n, denom):
        if denom == 0 or pd.isna(denom):
            return f"{n} (0.0%)"
        return f"{int(n)} ({(n / denom) * 100:.1f}%)"

    def fmt_mean_sd(x):
        x = pd.to_numeric(x, errors="coerce").dropna()
        if len(x) == 0:
            return ""
        return f"{x.mean():.2f} ({x.std():.2f})"

    def fmt_median_iqr(x):
        x = pd.to_numeric(x, errors="coerce").dropna()
        if len(x) == 0:
            return ""
        q1 = x.quantile(0.25)
        q3 = x.quantile(0.75)
        return f"{x.median():.1f} ({q1:.1f} to {q3:.1f})"

    def fmt_range(x):
        x = pd.to_numeric(x, errors="coerce").dropna()
        if len(x) == 0:
            return ""
        return f"{x.min():.1f} to {x.max():.1f}"

    def continuous_pvalue(sub, col):
        alive = pd.to_numeric(sub.loc[sub["died"] == 0, col], errors="coerce").dropna()
        death = pd.to_numeric(sub.loc[sub["died"] == 1, col], errors="coerce").dropna()

        if len(alive) < 2 or len(death) < 2:
            return "Not available"

        try:
            _, p = stats.mannwhitneyu(alive, death, alternative="two-sided")
        except Exception:
            return "Not available"

        return format_p(p, suffix="W")

    def categorical_pvalue(sub, col):
        temp = sub[[col, "died"]].dropna()

        if temp[col].nunique() < 2 or temp["died"].nunique() < 2:
            return "Not available"

        table = pd.crosstab(temp[col], temp["died"])

        try:
            chi2, p, dof, expected = stats.chi2_contingency(table)

            # Use Fisher only for 2x2 tables with small expected counts
            if table.shape == (2, 2) and (expected < 5).any():
                _, p = stats.fisher_exact(table)
                return format_p(p, suffix="E")

            return format_p(p, suffix="C")

        except Exception:
            return "Not available"

    def format_p(p, suffix=""):
        if pd.isna(p):
            return "Not available"
        if p < 0.001:
            return f"<0.001{suffix}"
        return f"{p:.3f}{suffix}"

    def get_group_df(group_name):
        if group_name == "All Obesity":
            return df[df["Obesity_Class"].notna()].copy()
        else:
            return df[df["Obesity_Class"] == group_name].copy()

    def group_counts(sub):
        n_all = len(sub)
        n_alive = (sub["died"] == 0).sum()
        n_death = (sub["died"] == 1).sum()
        return n_all, n_alive, n_death

    # Table structure
    group_names = [
        "All Obesity",
        "Class I Obesity",
        "Class II Obesity",
        "Class III Obesity"
    ]

    rows = []

    def add_empty_section(section_name):
        row = {
            "Variable": section_name,
            "Statistic or Category": ""
        }
        for g in group_names:
            row[(g, "All")] = ""
            row[(g, "Alive")] = ""
            row[(g, "Death")] = ""
            row[(g, "P value")] = ""
        rows.append(row)

    def add_continuous_variable(label, col):
        if col not in df.columns:
            return

        stats_rows = [
            ("Mean (SD)", fmt_mean_sd),
            ("Median (Q1 to Q3)", fmt_median_iqr),
            ("Range", fmt_range)
        ]

        first_row = True

        for stat_name, stat_func in stats_rows:
            row = {
                "Variable": label if first_row else "",
                "Statistic or Category": stat_name
            }

            for g in group_names:
                sub = get_group_df(g)
                alive = sub[sub["died"] == 0]
                death = sub[sub["died"] == 1]

                row[(g, "All")] = stat_func(sub[col])
                row[(g, "Alive")] = stat_func(alive[col])
                row[(g, "Death")] = stat_func(death[col])

                if first_row:
                    row[(g, "P value")] = continuous_pvalue(sub, col)
                else:
                    row[(g, "P value")] = ""

            rows.append(row)
            first_row = False

    def add_categorical_variable(label, col, categories=None):
        if col not in df.columns:
            return

        if categories is None:
            categories = [x for x in df[col].dropna().unique()]

        first_row = True

        for cat in categories:
            row = {
                "Variable": label if first_row else "",
                "Statistic or Category": cat
            }

            for g in group_names:
                sub = get_group_df(g)
                alive = sub[sub["died"] == 0]
                death = sub[sub["died"] == 1]

                n_all, n_alive, n_death = group_counts(sub)

                row[(g, "All")] = fmt_n_pct((sub[col] == cat).sum(), n_all)
                row[(g, "Alive")] = fmt_n_pct((alive[col] == cat).sum(), n_alive)
                row[(g, "Death")] = fmt_n_pct((death[col] == cat).sum(), n_death)

                if first_row:
                    row[(g, "P value")] = categorical_pvalue(sub, col)
                else:
                    row[(g, "P value")] = ""

            rows.append(row)
            first_row = False

    def add_binary_condition(label, col, section_label=None):
        if col not in df.columns:
            return

        row = {
            "Variable": section_label if section_label else "",
            "Statistic or Category": label
        }

        for g in group_names:
            sub = get_group_df(g)
            alive = sub[sub["died"] == 0]
            death = sub[sub["died"] == 1]

            n_all, n_alive, n_death = group_counts(sub)

            row[(g, "All")] = fmt_n_pct((sub[col] == 1).sum(), n_all)
            row[(g, "Alive")] = fmt_n_pct((alive[col] == 1).sum(), n_alive)
            row[(g, "Death")] = fmt_n_pct((death[col] == 1).sum(), n_death)
            row[(g, "P value")] = categorical_pvalue(sub, col)

        rows.append(row)

    # Build table rows

    # Continuous variables
    add_continuous_variable("Age", "INITIAL_AGE")
    add_continuous_variable("BMI", "Initial_BMI")

    # Demographics / functional categories
    add_categorical_variable(
        "Age Group, N (%)",
        "Age_Group_Table1",
        categories=["65-74", "75-84", ">=85"]
    )

    add_categorical_variable(
        "Gender, N (%)",
        "Gender_Table1",
        categories=["Male", "Female"]
    )

    add_categorical_variable(
        "Race, N (%)",
        "Race_Table1",
        categories=[
            "White",
            "Hispanic",
            "Black",
            "Asian, American Indian/Alaskan Native, Native Hawaiian/Pacific Islander",
            "Missing"
        ]
    )

    add_categorical_variable(
        "ADL Score, N (%)",
        "ADL_Category_Table1",
        categories=["0-12", "13-20", "21-28"]
    )

    add_categorical_variable(
        "Mobility, N (%)",
        "Mobility_Category_Table1",
        categories=["None", "Mild", "Moderate", "Severe"]
    )

    add_categorical_variable(
        "Cognitive Status, N (%)",
        "BIMS_Category_Table1",
        categories=[
            "Cognitively intact",
            "Mildly impaired",
            "Moderately impaired",
            "Severely impaired"
        ]
    )

    # Medical conditions
    add_empty_section("Medical conditions, N (%)")

    condition_map = [
        ("Arthritis", "ARTHRITIS"),
        ("Stroke", "STROKE"),
        ("Cancer", "CANCER"),
        ("COPD", "COPD"),
        ("Diabetes", "DIAB"),
        ("Dementia", "DEMENTIA"),
        ("Depression", "DEPRESSION"),
        ("Heart Failure", "HEARTFAILURE"),
        ("Hypertension", "HYPERTENSION"),
        ("End-Stage Renal Disease", "ESRD"),
        ("Fall History", "FT_PROBLEM"),
        ("Hearing Impairment", "HEARING"),
        ("Bowel Incontinence", "BOWEL_INCONTINENCE"),
        ("Urinary Incontinence", "URINE_INCONTINENCE"),
        ("Anxiety", "ANXTY"),
        ("Manic Depression", "MNC_DPRSN"),
        ("Schizophrenia", "SCHZOPRNIA"),
        ("Pressure Ulcer", "PRESSURE_ULCER")
    ]

    for label, col in condition_map:
        add_binary_condition(label, col)

    # Convert to dataframe
    table1_rows = []

    for row in rows:
        flat_row = {
            "Variable": row.get("Variable", ""),
            "Statistic or Category": row.get("Statistic or Category", "")
        }

        for g in group_names:
            for subcol in ["All", "Alive", "Death", "P value"]:
                flat_row[f"{g} - {subcol}"] = row.get((g, subcol), "")

        table1_rows.append(flat_row)

    table1 = pd.DataFrame(table1_rows)

    return table1


# Create Table 1-style table
table1_mortality_obesity = create_table1_mortality_by_obesity(summary_df)

display(table1_mortality_obesity)

# Optional save
table1_mortality_obesity.to_csv(
     "Visualizations_and_Tables/table1_mortality_by_obesity.csv",
     index=False
)

#table1_mortality_obesity.to_excel(
#    "Visualizations_and_Tables/table1_mortality_by_obesity.xlsx",
#    index=False
#)

,Variable,Statistic or Category,All Obesity - All,All Obesity - Alive,All Obesity - Death,All Obesity - P value,Class I Obesity - All,Class I Obesity - Alive,Class I Obesity - Death,Class I Obesity - P value,Class II Obesity - All,Class II Obesity - Alive,Class II Obesity - Death,Class II Obesity - P value,Class III Obesity - All,Class III Obesity - Alive,Class III Obesity - Death,Class III Obesity - P value
0,Age,Mean (SD),79.06 (8.20),77.01 (7.79),81.15 (8.09),<0.001W,80.23 (8.75),77.98 (8.23),82.50 (8.69),<0.001W,78.86 (7.57),77.85 (7.89),79.81 (7.21),0.180W,76.09 (6.54),73.54 (5.21),79.10 (6.73),<0.001W
1,,Median (Q1 to Q3),78.0 (72.0 to 86.0),75.0 (70.0 to 83.0),81.0 (75.0 to 88.0),,80.0 (73.0 to 87.0),77.0 (71.0 to 85.8),83.0 (76.0 to 89.0),,78.5 (72.0 to 84.0),77.0 (72.0 to 84.0),80.0 (74.0 to 86.0),,75.0 (71.0 to 80.0),73.0 (69.2 to 78.0),78.0 (74.0 to 84.5),
2,,Range,66.0 to 101.0,66.0 to 96.0,66.0 to 101.0,,66.0 to 101.0,66.0 to 96.0,66.0 to 101.0,,66.0 to 95.0,66.0 to 94.0,66.0 to 95.0,,66.0 to 91.0,66.0 to 84.0,67.0 to 91.0,
3,BMI,Mean (SD),36.13 (5.91),36.39 (6.17),35.87 (5.63),0.423W,32.15 (1.35),32.21 (1.34),32.09 (1.36),0.480W,37.09 (1.37),37.37 (1.44),36.83 (1.25),0.054W,45.90 (5.35),45.96 (5.94),45.84 (4.64),0.581W
4,,Median (Q1 to Q3),34.1 (31.8 to 38.3),34.0 (32.1 to 38.9),34.2 (31.6 to 37.9),,32.1 (31.0 to 33.2),32.2 (31.0 to 33.2),31.8 (30.9 to 33.2),,37.1 (35.9 to 38.1),37.3 (36.2 to 38.4),36.6 (35.7 to 37.9),,44.4 (41.9 to 49.2),43.4 (41.4 to 49.2),44.9 (42.4 to 48.9),
5,,Range,30.0 to 66.0,30.0 to 66.0,30.0 to 56.3,,30.0 to 35.0,30.0 to 34.9,30.0 to 35.0,,35.1 to 40.0,35.1 to 40.0,35.1 to 39.9,,40.0 to 66.0,40.0 to 66.0,40.1 to 56.3,
6,"Age Group, N (%)",65-74,149 (34.7%),96 (44.2%),53 (24.9%),<0.001C,72 (30.6%),48 (40.7%),24 (20.5%),0.002C,39 (35.5%),22 (41.5%),17 (29.8%),0.409C,38 (44.7%),26 (56.5%),12 (30.8%),<0.001C
7,,75-84,161 (37.4%),78 (35.9%),83 (39.0%),,80 (34.0%),38 (32.2%),42 (35.9%),,44 (40.0%),20 (37.7%),24 (42.1%),,37 (43.5%),20 (43.5%),17 (43.6%),
8,,>=85,120 (27.9%),43 (19.8%),77 (36.2%),,83 (35.3%),32 (27.1%),51 (43.6%),,27 (24.5%),11 (20.8%),16 (28.1%),,10 (11.8%),0 (0.0%),10 (25.6%),
9,"Gender, N (%)",Male,125 (29.1%),49 (22.6%),76 (35.7%),0.004C,75 (31.9%),29 (24.6%),46 (39.3%),0.022C,28 (25.5%),12 (22.6%),16 (28.1%),0.664C,22 (25.9%),8 (17.4%),14 (35.9%),0.091C
